In [1]:
import urllib.error
import urllib.request
import pprint
from langchain.tools import tool

from langchain.chat_models import init_chat_model

import langchain_groq
import os

from dotenv import load_dotenv

load_dotenv()


False

In [ ]:

model = init_chat_model("llama-3.3-70b-versatile",
                        api_key=os.environ["GROQ_API_KEY"],
                        model_provider="groq",
                        # base_url="https://api.groq.com/openai/v1",
                        max_tokens=1000, temperature=0.0)

# model = init_chat_model("openai/gpt-4o-mini",
#                         api_key=os.environ["OPENROUTER_API_KEY"],
#                         model_provider="openrouter",
#                         base_url="https://openrouter.ai/api/v1",
#                         max_tokens=1000, temperature=0.0)

# model = init_chat_model("nvidia/nemotron-3-ultra-550b-a55b:free",
#                         api_key=os.environ["OPENROUTER_API_KEY"],
#                         model_provider="openrouter",
#                         base_url="https://openrouter.ai/api/v1",
#                         max_tokens=1000, temperature=0.0)


# model = init_chat_model("openrouter/free",
#                         api_key=os.environ["OPENROUTER_API_KEY"],
#                         model_provider="openrouter",
#                         base_url="https://openrouter.ai/api/v1",
#                         max_tokens=1000, temperature=0.0)




In [8]:
model_or = init_chat_model("openrouter/free",
                        api_key=os.environ["OPENROUTER_API_KEY"],
                        model_provider="openrouter",
                        base_url="https://openrouter.ai/api/v1",
                        max_tokens=1000, temperature=0.0)


In [3]:
response = model.invoke("which model are you?")

pprint.pprint("Model response:")
pprint.pprint(response)

'Model response:'
AIMessage(content='I am a Meta AI model, and my specific model name is Llama. Llama stands for "Large Language Model Meta AI."', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 28, 'prompt_tokens': 40, 'total_tokens': 68, 'completion_time': 0.102529628, 'completion_tokens_details': None, 'prompt_time': 0.00572519, 'prompt_tokens_details': None, 'queue_time': 0.163170586, 'total_time': 0.108254818}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_3272ea2d91', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019f9748-c0ee-71d0-9b3e-388fd51df602-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 40, 'output_tokens': 28, 'total_tokens': 68})


# Structured Output

In [4]:
booking_requests = [
    "Hi, I'd like 2 tickets for Interstellar at the 7pm show tonight, name is Priya.",
    "can u book me a seat for the 9:30 showing of dune part two? im rohan",
    "URGENT - need to CANCEL my booking for Oppenheimer, confirmation was under Aisha",
]


In [5]:

for msg in booking_requests:
    r = model.invoke(f"Extract the customer's name, movie, and what they want (book or cancel) from: {msg}")
    print(r.content)
    print("---")

Here are the extracted details:

1. **Customer's name**: Priya
2. **Movie**: Interstellar
3. **What they want**: Book (they want to book 2 tickets)
---
Here are the extracted details:

* Customer's name: Rohan
* Movie: Dune Part Two
* What they want: Book a seat (for the 9:30 showing)
---
Here are the extracted details:

* Customer's name: Aisha
* Movie: Oppenheimer
* What they want: Cancel their booking
---


In [ ]:

for msg in booking_requests:
    r = model_or.invoke(f"Extract the customer's name, movie, and what they want (book or cancel) from: {msg}")
    print(r.content)
    print("---")

Customer's Name: Priya  
Movie: Interstellar  
Request: Book
---
Customer's Name: Rohan  
Movie: Dune Part Two  
Request: Book
---
Customer's Name: Aisha  
Movie: Oppenheimer  
Action: Cancel
---


In [9]:
from pydantic import BaseModel, Field
from typing import Literal

class BookingRequest(BaseModel):
    customer_name: str = Field(description="The customer's name")
    movie_title: str = Field(description="The movie they want to see")
    action: Literal["book", "cancel"] = Field(description="Whether this is a new booking or a cancellation")
    ticket_count: int = Field(description="How many tickets, default 1 if not mentioned", default=1)



In [11]:
model_or_with_str_putput = model_or.with_structured_output(BookingRequest)


In [12]:
for msg in booking_requests:
    r = model_or_with_str_putput.invoke(f"Extract a booking request from: {msg}")
    print(r)
    print(f" --> action type : {type(r.action)}, value : {r.action}")
    print("---")

customer_name='Priya' movie_title='Interstellar' action='book' ticket_count=2
 --> action type : <class 'str'>, value : book
---
customer_name='Rohan' movie_title='Dune Part Two' action='book' ticket_count=1
 --> action type : <class 'str'>, value : book
---
customer_name='Aisha' movie_title='Oppenheimer' action='cancel' ticket_count=1
 --> action type : <class 'str'>, value : cancel
---


# Tool Strategy & Provider Strategy

Two different mechanisms achieve the same guarantee. ProviderStrategy uses the model provider's own native structured-output feature (fast, but only works where supported). ToolStrategy fakes it via a synthetic tool call (works almost everywhere, slightly slower).




In [13]:
from langchain.agents.structured_output import ProviderStrategy, ToolStrategy

In [14]:
provider_strategy_model = model.with_structured_output(BookingRequest, strategy=ProviderStrategy(BookingRequest))

In [18]:
model.profile

{'name': 'Llama 3.3 70B Versatile',
 'release_date': '2024-12-06',
 'last_updated': '2024-12-06',
 'open_weights': True,
 'max_input_tokens': 131072,
 'max_output_tokens': 32768,
 'text_inputs': True,
 'image_inputs': False,
 'audio_inputs': False,
 'video_inputs': False,
 'text_outputs': True,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': False,
 'tool_calling': True,
 'attachment': False,
 'temperature': True}

In [24]:
model_or_35_turbo = init_chat_model("openai/gpt-3.5-turbo-0613",
                        api_key=os.environ["OPENROUTER_API_KEY"],
                        model_provider="openrouter",
                        base_url="https://openrouter.ai/api/v1",
                        max_tokens=1000, temperature=0.0)


In [25]:
model_or_35_turbo.profile

{'name': 'GPT-3.5 Turbo (older v0613)',
 'release_date': '2024-01-25',
 'last_updated': '2024-01-25',
 'open_weights': False,
 'max_input_tokens': 4095,
 'max_output_tokens': 4096,
 'text_inputs': True,
 'image_inputs': False,
 'audio_inputs': False,
 'video_inputs': False,
 'text_outputs': True,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': False,
 'tool_calling': True,
 'structured_output': True,
 'attachment': False,
 'temperature': True,
 'tool_call_streaming': True}

# Tool Strategy & Provider Strategy

Two different mechanisms achieve the same guarantee. `ProviderStrategy` uses the model
provider's own native structured-output feature (fast, but only works where supported).
`ToolStrategy` fakes it via a synthetic tool call (works almost everywhere, slightly slower).

In [26]:
from pydantic import BaseModel, Field
from typing import Literal
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy


class MeetingAction(BaseModel):
    """Action items extracted from a meeting transcript."""
    task: str = Field(description="The specific task to be completed")
    assignee: str = Field(description="Person responsible for the task")
    priority: Literal["low", "medium", "high"] = Field(description="Priority level")

agent = create_agent(
    model=model_or_35_turbo,
    tools=[],
    response_format=ToolStrategy(
        schema=MeetingAction,
        tool_message_content="Action item captured and added to meeting notes!"
    )
)

agent.invoke({
    "messages": [{"role": "user", "content": "From our meeting: Sarah needs to update the project timeline as soon as possible"}]
})

{'messages': [HumanMessage(content='From our meeting: Sarah needs to update the project timeline as soon as possible', additional_kwargs={}, response_metadata={}, id='aba7713e-4aa0-45ac-9536-999841ede75c'),
  AIMessage(content='', additional_kwargs={}, response_metadata={'model_name': 'openai/gpt-3.5-turbo-0613', 'id': 'gen-1784953055-ueBf1I0TK8CyfI0T35sh', 'created': 1784953055, 'object': 'chat.completion', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'openrouter', 'cost': 0.000151, 'cost_details': {'upstream_inference_completions_cost': 5.6e-05, 'upstream_inference_prompt_cost': 9.5e-05, 'upstream_inference_cost': 0.000151}, 'system_fingerprint': 'fp_20b2a3c29b'}, id='lc_run--019f977e-1f65-7bb3-9989-2d176cc5fbcd-0', tool_calls=[{'name': 'MeetingAction', 'args': {'task': 'Update the project timeline', 'assignee': 'Sarah', 'priority': 'high'}, 'id': 'call_C762ouwscURnerxlAAYSYEP3', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 95, '

In [27]:
from langchain_core.tools import tool

@tool
def peek_showtimes(movie_title: str) -> str:
    """Check showtimes for a movie."""
    return "7:00 PM and 10:15 PM"

In [28]:
incomplete_model = model_or_35_turbo.bind_tools([peek_showtimes]).with_structured_output(BookingRequest)

In [30]:
response=incomplete_model.invoke("Book 2 tickets for Interstellar at the 7pm show tonight, name is Priya. Also, check if there are any other showtimes available for Interstellar.")

In [31]:
response

BookingRequest(customer_name='Priya', movie_title='Interstellar', action='book', ticket_count=2)

In [36]:
from langchain.agents import create_agent

booking_agent = create_agent(
    model=model, 
    tools=[peek_showtimes],
    response_format=BookingRequest,
)

In [37]:
response=booking_agent.invoke({
    "messages": [{"role": "user", "content": "Is Interstellar showing tonight? Book 2 seats for Rohan"}]
})

In [38]:
response

{'messages': [HumanMessage(content='Is Interstellar showing tonight? Book 2 seats for Rohan', additional_kwargs={}, response_metadata={}, id='2229e60d-e0e1-4a9d-98a4-0eee6196bd15'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'shp73yq34', 'function': {'arguments': '{"movie_title":"Interstellar"}', 'name': 'peek_showtimes'}, 'type': 'function'}, {'id': 'qrffzcq6z', 'function': {'arguments': '{"action":"book","customer_name":"Rohan","movie_title":"Interstellar","ticket_count":2}', 'name': 'BookingRequest'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 51, 'prompt_tokens': 360, 'total_tokens': 411, 'completion_time': 0.092177485, 'completion_tokens_details': None, 'prompt_time': 0.038190833, 'prompt_tokens_details': None, 'queue_time': 0.05200104, 'total_time': 0.130368318}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'mod

In [39]:
class NewBooking(BaseModel):
    """A request to book NEW tickets."""
    customer_name: str
    movie_title: str
    ticket_count: int

class CancelBooking(BaseModel):
    """A request to CANCEL an existing booking."""
    customer_name: str
    movie_title: str

In [40]:
from typing import Union
union_agent = create_agent(
    model=model,
    tools=[peek_showtimes],
    response_format=Union[NewBooking, CancelBooking]
)

In [42]:
response=union_agent.invoke({"messages":[{"role":"user","content":"I want to cancel my movie Oppen"}]})

In [43]:
response

{'messages': [HumanMessage(content='I want to cancel my movie Oppen', additional_kwargs={}, response_metadata={}, id='a6a9ce77-d2c4-484f-81ed-5e1770dc14c0'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'dbq6btsw1', 'function': {'arguments': '{"customer_name":"I","movie_title":"Oppen"}', 'name': 'CancelBooking'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 21, 'prompt_tokens': 380, 'total_tokens': 401, 'completion_time': 0.057176267, 'completion_tokens_details': None, 'prompt_time': 0.048820165, 'prompt_tokens_details': None, 'queue_time': 0.058340814, 'total_time': 0.105996432}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019f978c-a9fd-7f71-9c3f-71c9f5be3117-0', tool_calls=[{'name': 'CancelBooking', 'args': {'customer_name': 'I', 'movie_title': 'Oppen'}, 'id': 'dbq6btsw1', 'type':

In [44]:
from typing import Union
union_agent_tool_Strategy = create_agent(
    model=model_or_35_turbo,
    tools=[peek_showtimes],
    response_format=ToolStrategy(Union[NewBooking, CancelBooking])
)

In [ ]:
response1=union_agent_tool_Strategy.invoke({"messages":[{"role":"user","content":"I want to cancel my movie Open"}]})

In [46]:
response1

{'messages': [HumanMessage(content='I want to cancel my movie Oppen', additional_kwargs={}, response_metadata={}, id='100d4bb5-a633-44b1-80e7-c3197b13225a'),
  AIMessage(content='', additional_kwargs={}, response_metadata={'model_name': 'openai/gpt-3.5-turbo-0613', 'id': 'gen-1784954146-OIdUV5PsteIQQbclqlsW', 'created': 1784954146, 'object': 'chat.completion', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'openrouter', 'cost': 0.000158, 'cost_details': {'upstream_inference_completions_cost': 4.4e-05, 'upstream_inference_prompt_cost': 0.000114, 'upstream_inference_cost': 0.000158}, 'system_fingerprint': 'fp_20b2a3c29b'}, id='lc_run--019f978e-c409-7432-ad63-5dee73046cec-0', tool_calls=[{'name': 'CancelBooking', 'args': {'customer_name': 'user', 'movie_title': 'Oppen'}, 'id': 'call_VDtn5kclWvbQe1dHyi9p5IUh', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 114, 'output_tokens': 22, 'total_tokens': 136, 'input_token_details': {'cache_read':

In [47]:
class SeatBooking(BaseModel):
    customer_name: str
    ticket_count: int = Field(description="Number of tickets, must be between 1 and 10", ge=1, le=10)

In [48]:
seat_agent= create_agent(
    model=model_or_35_turbo,
    tools=[],
    response_format=ToolStrategy(SeatBooking),
    system_prompt= "Extract the booking details exactly as stated, Don't invent anything"
)

In [49]:
result = seat_agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "Hi I am Mayank, I want to give party to my students, book 15 tickets"
        }
    ]
})

In [50]:
result

{'messages': [HumanMessage(content='Hi I am Mayank, I want to give party to my students, book 15 tickets', additional_kwargs={}, response_metadata={}, id='742901cb-2963-4596-807a-0198498ee805'),
  AIMessage(content='', additional_kwargs={}, response_metadata={'model_name': 'openai/gpt-3.5-turbo-0613', 'id': 'gen-1784954731-jSz8jH8GccDfsBVBdFcs', 'created': 1784954731, 'object': 'chat.completion', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'openrouter', 'cost': 0.000145, 'cost_details': {'upstream_inference_completions_cost': 4.6e-05, 'upstream_inference_prompt_cost': 9.9e-05, 'upstream_inference_cost': 0.000145}, 'system_fingerprint': 'fp_20b2a3c29b'}, id='lc_run--019f9797-b0a3-7651-93a9-8332c103bed5-0', tool_calls=[{'name': 'SeatBooking', 'args': {'customer_name': 'Mayank', 'ticket_count': 10}, 'id': 'call_9BhoJssVGMEUo4SqNVcocRXz', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 99, 'output_tokens': 23, 'total_tokens': 122, 'input

In [51]:
model_ollama = init_chat_model("ollama:llama3.1:8b")

In [57]:
model_ollama

ChatOllama(metadata={'lc_versions': {'langchain-core': '1.4.9', 'langchain': '1.3.13'}}, output_version=None, model='llama3.1:8b')

In [52]:
seat_agent= create_agent(
    model=model_ollama,
    tools=[],
    response_format=ToolStrategy(SeatBooking),
    system_prompt= "Extract the booking details exactly as stated, Don't invent anything"
)

In [53]:
result_ollama = seat_agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "Hi I am Mayank, I want to give party to my students, book 15 tickets"
        }
    ]
})

In [54]:

result_ollama

{'messages': [HumanMessage(content='Hi I am Mayank, I want to give party to my students, book 15 tickets', additional_kwargs={}, response_metadata={}, id='aa30f502-b1b4-4731-aaaa-6ed0035ba1c9'),
  AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'llama3.1:8b', 'created_at': '2026-07-25T04:50:24.7979285Z', 'done': True, 'done_reason': 'stop', 'total_duration': 60247648000, 'load_duration': 42391614500, 'prompt_eval_count': 203, 'prompt_eval_duration': 15578128000, 'eval_count': 26, 'eval_duration': 2261955000, 'logprobs': None, 'model_name': 'llama3.1:8b', 'model_provider': 'ollama'}, id='lc_run--019f979b-4a43-76f2-9860-d72c82837d3b-0', tool_calls=[{'name': 'SeatBooking', 'args': {'customer_name': 'Mayank', 'ticket_count': 15}, 'id': 'bc3cc7de-5893-452a-b9cc-8e97c0dcc690', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 203, 'output_tokens': 26, 'total_tokens': 229}),
  ToolMessage(content="Error: Failed to parse structured output for